# Discussion 1 Assignment

Alexander Zhang  
alexacz1@uci.edu

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import requests
import zipfile

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, RocCurveDisplay
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
def load_diabetes_readmission_data() -> pd.DataFrame:
    url = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
    zip_filepath = "diabetes_readmission.zip"
    extract_folder = "diabetes_data_extracted"
    
    if not os.path.exists(zip_filepath):
        print(f"Downloading dataset...")
        r = requests.get(url)
        with open(zip_filepath, "wb") as f:
            f.write(r.content)
            
    with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
        zip_ref.extractall(extract_folder)

    csv_path = None
    for root, dirs, files in os.walk(extract_folder):
        for file in files:
            if file.lower() == "diabetic_data.csv":
                csv_path = os.path.join(root, file)
                break
    
    if csv_path:
        print(f"Found file at: {csv_path}")
        return pd.read_csv(csv_path)
    else:
        raise FileNotFoundError("Could not find diabetic_data.csv in the extracted files.")

df = load_diabetes_readmission_data()
print(f"Success! Data loaded with {df.shape[0]} rows.")

Found file at: diabetes_data_extracted/diabetic_data.csv
Success! Data loaded with 101766 rows.


## Data Summarization
We are now going to summarize the data and preprocess it in a simple manner.  
It will be framed as a binary classification problem: Did the patient get readmitted in less than 30 days?  
We will use 1 for Yes (Early Readmission), 0 = No (or readmitted after 30 days).  
The data chosen will be a few intuitive, fully-populated numeric columns.

In [6]:
print("\n--- DATA SUMMARIZATION ---")
print(f"Total number of patients/records: {len(df)}")
print(f"Total number of variables: {df.shape[1]}")

df['target'] = (df['readmitted'] == '<30').astype(int)
print("\nTarget Variable Distribution (0 = No Early Readmission, 1 = Early Readmission):")
print(df['target'].value_counts(normalize=True) * 100)

features = [
    'time_in_hospital', 
    'num_lab_procedures', 
    'num_medications', 
    'number_diagnoses'
]

X = df[features]
y = df['target']


--- DATA SUMMARIZATION ---
Total number of patients/records: 101766
Total number of variables: 50

Target Variable Distribution (0 = No Early Readmission, 1 = Early Readmission):
target
0    88.840084
1    11.159916
Name: proportion, dtype: float64
